In [1]:
from datasets import load_dataset

ds = load_dataset("meta-math/GSM8K_zh")

/home/u12321044/anaconda3/envs/baichuan/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 8792 examples [00:00, 31953.67 examples/s]


In [2]:
from datasets import load_dataset
import numpy as np

# 加载数据集
dataset = load_dataset("openai/gsm8k", "main")

# 获取问题和解答字段
questions = dataset['train']['question']
answers = dataset['train']['answer']

# 计算样本数量
sample_count = len(questions)

# 计算问题和答案的长度（这里以单词计数为例）
question_lengths = [len(q.split()) for q in questions]
answer_lengths = [len(a.split()) for a in answers]

# 基础统计信息
avg_question_length = np.mean(question_lengths)
min_question_length = min(question_lengths)
max_question_length = max(question_lengths)

avg_answer_length = np.mean(answer_lengths)
min_answer_length = min(answer_lengths)
max_answer_length = max(answer_lengths)

# 计算独特词汇量
vocab = set()
for text in questions + answers:
    vocab.update(text.split())
vocab_size = len(vocab)

# 打印结果
print(f"样本数量: {sample_count}")
print(f"问题长度 - 平均: {avg_question_length}, 最小: {min_question_length}, 最大: {max_question_length}")
print(f"解答长度 - 平均: {avg_answer_length}, 最小: {min_answer_length}, 最大: {max_answer_length}")
print(f"独特词汇量: {vocab_size}")

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 139475.28 examples/s]


样本数量: 7473
问题长度 - 平均: 45.092600026763016, 最小: 9, 最大: 183
解答长度 - 平均: 51.71176234443998, 最小: 4, 最大: 216
独特词汇量: 50546


In [3]:

# 查看训练集的前5条数据
for i in range(5):
    print(f"样本 {i+1}:")
    print(f"问题: {dataset['train'][i]['question']}")
    print(f"解答: {dataset['train'][i]['answer']}")
    print("-"*50)  # 分隔线，便于区分不同样本

样本 1:
问题: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
解答: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72
--------------------------------------------------
样本 2:
问题: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
解答: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10
--------------------------------------------------
样本 3:
问题: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
解答: In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents 

In [2]:
# 转为sft格式
import json

# 原始数据文件路径和目标文件路径
input_file = "/home/u12321044/share/liang_52/align_tax/gsm8k_aligned_answers_vllm.jsonl"  # 原始数据文件
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k.json"  # 转换后的数据文件

# 读取原始数据
with open(input_file, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

# 转换数据格式
formatted_data = []
for sample in raw_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["original_answer"]  # output 对应 original_answer
    }
    formatted_data.append(formatted_sample)

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k.json


In [3]:
# 生成test数据
from datasets import load_dataset
import json

# 加载GSM8K数据集
dataset = load_dataset("openai/gsm8k", "main")

# 提取测试集
test_data = dataset['test']

# 转换数据格式
formatted_data = []
for sample in test_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["answer"]  # output 对应 answer, 注意这里直接使用了"answer"字段，如果你的数据中有"original_answer"请相应替换
    }
    formatted_data.append(formatted_sample)

# 目标文件路径
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_test.json"

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

/home/u12321044/anaconda3/envs/baichuan/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_test.json


In [4]:
# 转为self-style-sft格式
import json

# 原始数据文件路径和目标文件路径
input_file = "/home/u12321044/share/liang_52/align_tax/gsm8k_aligned_answers_vllm.jsonl"  # 原始数据文件
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_align.json"  # 转换后的数据文件

# 读取原始数据
with open(input_file, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

# 转换数据格式
formatted_data = []
for sample in raw_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["aligned_answer"]  # output 对应 original_answer
    }
    formatted_data.append(formatted_sample)

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_align.json


In [1]:
from transformers import TrainingArguments

# 指定 training_args.bin 文件所在的路径
training_args_path = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/saves/Qwen2.5-1.5B-Instruct/full/train_2025-03-17-22-39-gsm8k/training_args.bin"

# 加载 training_args.bin 文件
training_args = TrainingArguments(training_args_path)

# 打印加载的参数
print(training_args)

/home/u12321044/anaconda3/envs/baichuan/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TrainingArguments(
_n_gpu=6,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=no,
eval_use_gather_object=F